# Intent Benchmark — run every intent pipeline on the arena's test sets

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/OpenVoiceOS/ngi0-notebooks/blob/dev/benchmarks/intent_benchmark.ipynb) [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenVoiceOS/ngi0-notebooks/blob/dev/benchmarks/intent_benchmark.ipynb)

## Start here

This notebook installs the OpenVoiceOS intent pipeline plugins and runs every intent fighter of the [OVOS plugin arena](https://github.com/OpenVoiceOS/ovos-plugin-arena) on the arena's intent test sets. A **fighter** is one pipeline: one or more intent plugins in a fixed order, with fixed settings. For each test set and language, the notebook trains the fighter on the training data of that test set, then gives it every test sentence. A row records the intent the fighter chose, the correct intent and the time it took.

The notebook uses the arena's own code to train, to predict and to score, so its numbers are the numbers the arena shows. It also counts the correct answers itself from the rows, and says if its count differs.

Run the cells from top to bottom. The default settings run 36 fighters on the 12 intent test sets of the arena, 1332 fighter runs (fighter, test set and language) in all. They leave out the 7 fighters that use `ovos-jurebes-pipeline-plugin`; set `EXCLUDE_PLUGINS = []` to run them too. A full run takes hours on one computer. Set `SMOKE = True` for a run of a few minutes on one fighter.

No fighter needs an API key. Every plugin runs on your computer.

In [ ]:
#@title Code that reads the settings
# A setting comes from an environment variable of the same name, else from a line NAME=value in the file .env,
# else from the value in the Settings cell. A value from outside is turned into the type of that value.
import json, os

_DOTENV = {}
if os.path.isfile(".env"):
    with open(".env", encoding="utf-8") as _f:
        for _line in _f:
            _name, _sep, _value = _line.strip().partition("=")
            if _sep and not _name.startswith("#"):
                _DOTENV[_name.strip()] = _value.strip().strip("\"'")


def setting(name, default):
    raw = os.environ.get(name, _DOTENV.get(name))
    if raw is None:
        return default
    if isinstance(default, bool):
        if raw.strip().lower() in ("1", "true", "yes", "on"):
            return True
        if raw.strip().lower() in ("0", "false", "no", "off", ""):
            return False
        raise ValueError(f"{name} must be True or False, not {raw!r}")
    if isinstance(default, (int, float)):
        try:
            return type(default)(raw)
        except ValueError:
            raise ValueError(f"{name} must be a number, not {raw!r}") from None
    if isinstance(default, (list, dict)):
        return json.loads(raw)
    return raw


In [ ]:
# Settings: change the values below.
DATASETS = setting("DATASETS", ["intents-for-eval", "ovos-intents-v5", "massive-templates", "banking77", "clinc150", "snips", "mtop-en-US", "mtop-de-DE", "mtop-es-ES", "mtop-fr-FR", "mtop-hi-IN", "mtop-th-TH"])   # arena test sets to run
LEAGUES = setting("LEAGUES", [])                      # leagues to run, such as ["intent_offline"]; empty: every league
FIGHTERS = setting("FIGHTERS", [])                    # fighters to run, such as ["padatious-medium"]; empty: every fighter of LEAGUES
EXCLUDE_PLUGINS = setting("EXCLUDE_PLUGINS", ["ovos-jurebes-pipeline-plugin"])   # a fighter with any of these plugins in its pipeline does not run
LANGS = setting("LANGS", [])                          # languages to run, such as ["en-US", "pt-PT"]; empty: every language of each test set
MAX_SAMPLES = setting("MAX_SAMPLES", 0)               # test sentences per test set and language; 0: all of them
RUN = setting("RUN", True)                            # False: skip step 3 and score the rows already in OUTPUT_DIR
UPLOAD = setting("UPLOAD", False)                     # True: send the results to the arena's Hugging Face datasets
HF_TOKEN = setting("HF_TOKEN", "")                    # a Hugging Face token with write access; only for UPLOAD

# Quick test
SMOKE = setting("SMOKE", False)                       # True: one fighter, one test set, a few sentences; nothing is uploaded
N_SAMPLES = setting("N_SAMPLES", 40)                  # quick test only: test sentences
# EPOCHS is not used: the fighters train with their own settings.

# ── Advanced: leave these as they are unless you know you need them ──
TRAIN_TIMEOUT_SECS = setting("TRAIN_TIMEOUT_SECS", 1200)   # stop a fighter that trains and predicts for longer than this; 0: no limit
TRAIN_MAX_RSS_MB = setting("TRAIN_MAX_RSS_MB", 8192)       # stop a fighter that uses more memory than this; 0: no limit
ARENA_REF = setting("ARENA_REF", "c59165f3b528c7329d6cb4999bef6cc75db770a3")   # the plugin arena commit whose registry, runner and scoring code are used
WORK_DIR = setting("WORK_DIR", "")                    # the folder for downloads; empty: ./intent_benchmark_work, on Kaggle /tmp/intent_benchmark_work
OUTPUT_DIR = setting("OUTPUT_DIR", "./intent_benchmark_output")   # the result rows and the tables go here
SHOW_DETAILS = setting("SHOW_DETAILS", False)         # True: also print the tables behind every step

In [ ]:
import os

WORK_DIR = WORK_DIR or ("/tmp/intent_benchmark_work" if os.path.isdir("/kaggle") else "./intent_benchmark_work")
SMOKE_FIGHTERS = ["nebulacioso"]
if SMOKE:
    DATASETS, LANGS, MAX_SAMPLES, UPLOAD = ["intents-for-eval"], ["en-US"], N_SAMPLES, False
    OUTPUT_DIR = OUTPUT_DIR.rstrip("/") + "_smoke"   # the trial's few rows never mix with a real run's rows

## Step 1 — Install the plugins and get the arena code

Installs the intent pipeline plugins from PyPI, and downloads one fixed commit of the plugin arena: its list of fighters and test sets, the code that trains and runs a fighter, and the code that scores it. If a red box appears, read its last line.

In [ ]:
#@title Code for step 1: the step lines, the internet check and the arena checkout
import json, os, subprocess, sys, time, urllib.error, urllib.request
from pathlib import Path

def detail(*a, **k):
    # SHOW_DETAILS: a line of the tables behind a step, with the clock time
    if SHOW_DETAILS:
        print(time.strftime("%H:%M:%S"), *a, **k)
if not SHOW_DETAILS:
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # no download bars; the step lines say what happened
_STEPS, _last, _done = {}, [time.time()], [time.time()]
def tick(name):
    now = time.time()
    _STEPS[name], _last[0] = now - _last[0], now
def done(name, message):
    # the step's one line, with the clock time and the seconds the step took
    tick(name)
    print(f"{message} [{time.strftime('%H:%M:%S')}, {_last[0] - _done[0]:.0f} s]")
    _done[0] = _last[0]

def check_internet():
    try:
        urllib.request.urlopen("https://huggingface.co", timeout=15)
    except (urllib.error.URLError, OSError) as exc:
        raise SystemExit("No internet. On Kaggle, open the Settings panel and switch Internet on. "
                         f"({exc})") from None

ARENA_URL = "https://github.com/OpenVoiceOS/ovos-plugin-arena"

def fetch_arena(ref, folder):
    # one commit of the plugin arena: its registry, its runner and its scoring code
    folder = Path(folder)
    def git(*args, check=True):
        return subprocess.run(["git", "-C", str(folder), *args], check=check,
                              capture_output=True, text=True).stdout.strip()
    if not (folder / ".git").is_dir():
        folder.mkdir(parents=True, exist_ok=True)
        git("init", "-q")
        git("remote", "add", "origin", ARENA_URL)
    if git("rev-parse", "-q", "--verify", "HEAD", check=False) != ref:
        git("fetch", "-q", "--depth", "1", "origin", ref)
        git("checkout", "-q", "--force", "FETCH_HEAD")
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
    return git("rev-parse", "HEAD")


In [ ]:
check_internet()
%pip install --quiet --pre "ovos-plugin-manager>=2.0.0" "ovos-core>=3.2.0a1" "ovos-spec-tools>=1.11.0a1" "pydantic>=2" "huggingface_hub>=0.30" "datasets>=2.20" "pyarrow>=15" "pandas>=2" "ovos-padatious>=2.3.2a1" "padacioso>=2.5.4a1" "ovos-adapt-parser>=1.6.8a1" "nebulento>=0.10.6a1" "palavreado>=1.7.6a1" "linha-fina>=0.4.4a2" "jurebes>=0.4.1a1" "ovos-markov-pipeline-plugin>=0.5.2a2" "ovos-m2v-pipeline>=0.38.0a1"
ARENA_DIR = Path(WORK_DIR) / "ovos-plugin-arena"
ARENA_SHA = fetch_arena(ARENA_REF, ARENA_DIR)
from importlib.metadata import entry_points
INSTALLED = sorted({e.name for g in ("opm.pipeline",) for e in entry_points(group=g)})
detail("installed intent pipeline plugins:", INSTALLED)
done("install", f"Step 1 done: {len(INSTALLED)} intent pipeline plugins installed, plugin arena at commit {ARENA_SHA[:10]}.")

## Step 2 — Choose the fighters for each test set

Each test set has its own training data: example sentences, keywords or templates for each intent. A fighter runs on a test set only when every plugin in it can train on the training data of that test set. A keyword plugin, for example, cannot train where the test set has only example sentences. A fighter that trained on a test set does not run on it. A fighter whose plugins are not installed does not run, and the table says why.

In [ ]:
#@title Code for step 2: the fighters for each test set
import gc, logging
logging.getLogger().setLevel(logging.INFO if SHOW_DETAILS else logging.ERROR)

from registry.loaders import list_datasets, load_dataset
from importlib.metadata import PackageNotFoundError, version
from runner.intent_bench import eligible_competitors, predictions_store, resolve_revision, run_competitor_lang, UnresolvableModelPin
from runner.intent_pipeline import ENGINE_REGISTRY
from runner.queue_tools import is_trained_on

ALL_SETS = {d.dataset_id: d for d in list_datasets("intent")}

def train_sets(ds):
    return {paradigm: load_dataset(f"intent_{paradigm}", train_id)
            for paradigm, train_id in (ds.train_datasets or {}).items()}

def is_installed(dist):
    try:
        return bool(version(dist))
    except PackageNotFoundError:
        return False

def missing_plugins(comp):
    # the arena loads an engine by its import path, not by its entry-point name, so check its distribution
    return sorted({ENGINE_REGISTRY[p].dist for p in comp.pipeline_plugins if not is_installed(ENGINE_REGISTRY[p].dist)})

def fighters_for(ds):
    # ([fighter], [(fighter id, why it does not run)])
    chosen, skipped = [], []
    for comp in eligible_competitors(set(train_sets(ds)), ds.dataset_id):
        if SMOKE and comp.competitor_id not in SMOKE_FIGHTERS:
            continue
        if LEAGUES and comp.modality.value not in LEAGUES:
            continue
        if FIGHTERS and comp.competitor_id not in FIGHTERS:
            continue
        if excluded := sorted(set(comp.pipeline_plugins) & set(EXCLUDE_PLUGINS)):
            skipped.append((comp.competitor_id, f"left out by EXCLUDE_PLUGINS: {', '.join(excluded)}"))
        elif is_trained_on(comp, ds):
            skipped.append((comp.competitor_id, "trained on this test set"))
        elif missing := missing_plugins(comp):
            skipped.append((comp.competitor_id, f"not installed: {', '.join(missing)}"))
        else:
            chosen.append(comp)
    return chosen, skipped

def langs_for(ds, comp):
    wanted = ds.langs or [ds.lang]
    return [lang for lang in wanted if (not LANGS or lang in LANGS) and (not comp.langs or lang in comp.langs)]

In [ ]:
unknown = sorted(set(DATASETS) - set(ALL_SETS))
assert not unknown, f"DATASETS names {unknown}, which the arena does not know. Known: {sorted(ALL_SETS)}"
SETS = [ALL_SETS[d] for d in DATASETS]
CELLS, SKIPPED = [], []
for ds in SETS:
    chosen, skipped = fighters_for(ds)
    CELLS += [(ds, comp, lang) for comp in chosen for lang in langs_for(ds, comp)]
    SKIPPED += [(ds.dataset_id, cid, why) for cid, why in skipped]
    detail(ds.dataset_id, [c.competitor_id for c in chosen])
assert CELLS, "No fighter matches these test sets, LEAGUES and FIGHTERS."
done("fighters", f"Step 2 done: {len(CELLS)} fighter runs (fighter, test set and language) on {len(SETS)} test sets, {len(SKIPPED)} skipped.")

## Step 3 — Train and run the fighters

For each fighter, test set and language, the arena's runner trains the fighter in its own process, with a time limit and a memory limit, then gives it the test sentences one at a time. The result is one row per sentence. The rows go to `intent_benchmark_output/rows/`. A run that stops can start again: the rows already written are kept, and only the missing sentences run.

In [ ]:
#@title Code for step 3: one run per fighter, test set and language, with the arena's runner
ROWS = Path(OUTPUT_DIR) / "rows"
REVISIONS, TRAIN = {}, {}

def rows_path(ds, comp, lang):
    return ROWS / ds.dataset_id / predictions_store(comp) / lang / f"{comp.competitor_id}.jsonl"

def run_cell(ds, comp, lang):
    if ds.dataset_id not in REVISIONS:
        REVISIONS[ds.dataset_id] = resolve_revision(ds.source.hf_id, ds.source.revision)
        TRAIN[ds.dataset_id] = train_sets(ds)
    path = rows_path(ds, comp, lang)
    path.parent.mkdir(parents=True, exist_ok=True)
    written = run_competitor_lang(comp, ds.dataset_id, lang, ds, TRAIN[ds.dataset_id], REVISIONS[ds.dataset_id], path,
                                  max_samples=MAX_SAMPLES, transcript_cache_dir=Path(WORK_DIR) / "transcripts",
                                  train_timeout_secs=TRAIN_TIMEOUT_SECS, train_max_rss_mb=TRAIN_MAX_RSS_MB)
    gc.collect()
    return written

In [ ]:
RUNS, FAILED = {}, {}
for i, (ds, comp, lang) in enumerate(CELLS if RUN else [], 1):
    key, started = (ds.dataset_id, comp.competitor_id, lang), time.time()
    try:
        RUNS[key] = run_cell(ds, comp, lang)
    except UnresolvableModelPin as exc:   # the fighter's model cannot be found at its pinned revision
        FAILED[key] = str(exc)
    except Exception as exc:   # one broken plugin must not stop the others
        FAILED[key] = f"{type(exc).__name__}: {exc}"
    detail(f"{i}/{len(CELLS)}", *key, RUNS.get(key, FAILED.get(key)), f"{time.time() - started:.0f} s")
done("run", f"Step 3 done: {len(RUNS)} of {len(CELLS) if RUN else 0} runs finished, {len(FAILED)} failed.")

## Step 4 — Score the results

The arena's scoring code turns the rows into one board per league, test set and language, the same board the arena shows. The **accuracy** is the share of test sentences that got the correct intent. A sentence with no correct intent counts as correct when the fighter chose no intent. The boards rank on the **generalization accuracy**, in the column `score`: the accuracy on the test sentences that are not close to the training data. Higher is better. The notebook also counts the correct rows itself, from the intent the fighter chose and the correct intent of each row, and the column `own count agrees` says whether its count gives the same accuracy.

In [ ]:
#@title Code for step 4: one board per league, test set and language, and an own count
import pandas as pd
from arena.metrics import MIN_BOARD_SAMPLES, PRIMARY_METRIC, build_benchmark_board
from arena.predictions import read_jsonl

def own_correct(row, granularity):
    # the arena's rule, counted again from the prediction: a sentence with no intent is correct when the fighter chose none
    if row["reference_intent"] is None:
        return row["prediction"] is None
    if granularity == "domain":
        return row["prediction"] is not None and row["prediction"].split(":", 1)[0] == row["reference_intent"].split(":", 1)[0]
    return row["prediction"] == row["reference_intent"]

def own_accuracy(path, ds):
    # correct rows / all rows, counted from the raw rows without the arena's code
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    return sum(own_correct(r, ds.reference_granularity) for r in rows) / len(rows) if rows else None

def published_entry(league, ds, lang, competitor_id):
    path = ARENA_DIR / "frontend-static/public/data" / f"benchmark-{league}-{ds.dataset_id}-{lang}.json"
    if not path.is_file():
        return {}
    return next((e for e in json.loads(path.read_text())["entries"] if e["competitor_id"] == competitor_id), {})

def score(ds, league, lang):
    mine = {comp.competitor_id: comp for d, comp, l in CELLS if d is ds and l == lang and comp.modality.value == league}
    paths = {cid: rows_path(ds, comp, lang) for cid, comp in mine.items()}
    by_fighter = {cid: rows for cid, path in paths.items() if path.is_file() and (rows := read_jsonl(path))}
    if not by_fighter:
        return []
    board = build_benchmark_board(league, ds.dataset_id, lang, by_fighter, "notebook", dataset_revision=ds.source.revision)
    primary = PRIMARY_METRIC.get(league, "accuracy")
    table = []
    for e in board.entries:
        m, published, own = e.metrics, published_entry(league, ds, lang, e.competitor_id), own_accuracy(paths[e.competitor_id], ds)
        table.append({
            "league": league, "test set": ds.dataset_id, "lang": lang, "fighter": e.competitor_id,
            "sentences": e.samples, "score": m.get(primary), "metric": primary,
            "score low": e.primary_metric_ci_lower, "score high": e.primary_metric_ci_upper,
            "accuracy": m.get("accuracy"), "latency ms": m.get("latency_ms_median"),
            "ranked": not e.unranked, "why not ranked": e.unranked_reason or "",
            "own count agrees": own is not None and m.get("accuracy") is not None and round(own, 4) == round(m["accuracy"], 4),
            "published sentences": published.get("samples"),
            "published score": (published.get("metrics") or {}).get(primary),
        })
    return table

In [ ]:
GROUPS = sorted({(ds.dataset_id, comp.modality.value, lang) for ds, comp, lang in CELLS})
RESULTS = pd.DataFrame([row for d, league, lang in GROUPS for row in score(ALL_SETS[d], league, lang)])
assert len(RESULTS), "No fighter wrote a result row. Set SHOW_DETAILS = True and run again to see why."
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
RESULTS.to_csv(Path(OUTPUT_DIR) / "intent_benchmark.csv", index=False)
disagree = RESULTS[~RESULTS["own count agrees"]]
if SHOW_DETAILS:
    print(RESULTS.to_string(index=False))
done("score", f"Step 4 done: {len(RESULTS)} results on {len(GROUPS)} boards, {int(RESULTS['ranked'].sum())} ranked; "
     + (f"the own count differs for {list(zip(disagree['test set'], disagree['fighter']))}." if len(disagree) else "the own count agrees with every one."))

## Step 5 — Send the results to the arena (only with `UPLOAD = True`)

This step does nothing unless `UPLOAD` is `True`. Then it sends the rows to the arena's Hugging Face datasets, one dataset per test set and kind of training data. First it adds the rows that are already published and that this run did not make, so no published row is lost.

In [ ]:
#@title Code for step 5: merges the published rows into this run's rows, then uploads
from arena.predictions import fetch_hf_predictions
from registry.loaders import results_repo_for
from runner.intent_bench import upload_predictions

def merge_published(ds):
    # appends to each local file the published rows of the same fighter whose sentence is not in it
    added = 0
    for store_dir in sorted(d for d in (ROWS / ds.dataset_id).iterdir() if d.is_dir()):
        published = fetch_hf_predictions(results_repo_for(store_dir.name, ds.dataset_id))
        for path in store_dir.glob("*/*.jsonl"):
            remote = Path(published) / path.parent.name / path.name if published else None
            if not remote or not remote.is_file():
                continue
            have = {json.loads(line)["sample_id"] for line in path.read_text().splitlines() if line.strip()}
            extra = [line for line in remote.read_text().splitlines()
                     if line.strip() and json.loads(line)["sample_id"] not in have]
            with path.open("a") as fh:
                fh.writelines(line + "\n" for line in extra)
            added += len(extra)
    return added

In [ ]:
UPLOADED = []
if UPLOAD:
    if not HF_TOKEN and os.path.isdir("/kaggle"):
        try:
            from kaggle_secrets import UserSecretsClient
            HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
    from huggingface_hub import get_token
    HF_TOKEN = HF_TOKEN or get_token()   # the token that `hf auth login` stored
    assert HF_TOKEN, ("UPLOAD is True but there is no Hugging Face token. Set HF_TOKEN, add a Kaggle secret HF_TOKEN, "
                      "or run `hf auth login`. The token needs write access to the OpenVoiceOS organisation.")
    os.environ["HF_TOKEN"] = HF_TOKEN   # huggingface_hub reads the token from here
    for ds in SETS:
        if not (ROWS / ds.dataset_id).is_dir():
            continue
        added = merge_published(ds)
        print(f"Uploading {ds.dataset_id} ({added} published rows kept)")
        upload_predictions(ROWS / ds.dataset_id, ds.dataset_id, ds)
        UPLOADED.append(ds.dataset_id)
done("upload", "Step 5 done: " + (f"{len(UPLOADED)} test sets uploaded" if UPLOAD else "nothing uploaded, because UPLOAD is False") + ".")

## What you got

**The table** below lists the best result on each board. The whole table is in `intent_benchmark_output/intent_benchmark.csv`, and the result rows are in `intent_benchmark_output/rows/`.

Read the score together with the number of sentences. A result is ranked only when the score rests on 30 or more sentences; the column `why not ranked` gives the reason for the others. The columns `published sentences` and `published score` show what the arena published for the same fighter, so you can see how this run compares. Each league has its own boards: fighters that train on your sentences (`intent_online`), fighters with a model trained before (`intent_offline`), keyword fighters (`intent_keyword`) and fighters that do not train (`intent_zero_shot`).

<details>
<summary>Read more</summary>

- [The plugin arena](https://github.com/OpenVoiceOS/ovos-plugin-arena), its [methodology](https://github.com/OpenVoiceOS/ovos-plugin-arena/blob/dev/docs/methodology.md), and [how to add a fighter](https://github.com/OpenVoiceOS/ovos-plugin-arena/blob/dev/docs/add-a-fighter.md)
- [`docs/benchmarks.md`](../docs/benchmarks.md), the guide to these notebooks

</details>

In [ ]:
best = RESULTS.sort_values(["ranked", "score"], ascending=[False, False]).groupby(["league", "test set", "lang"]).head(1)
columns = ["league", "test set", "lang", "fighter", "sentences", "metric", "score"]
print(f"""Fighter runs          : {len(RUNS)} of {len(CELLS)} finished{', failed: ' + '; '.join('/'.join(k) for k in FAILED) if FAILED else ''}
Skipped               : {len(SKIPPED)}{' (' + '; '.join(f'{c} on {d}: {w}' for d, c, w in SKIPPED[:8]) + (' ...' if len(SKIPPED) > 8 else '') + ')' if SKIPPED else ''}
Results               : {len(RESULTS)}, {int(RESULTS['ranked'].sum())} ranked
Own count agrees      : {int(RESULTS['own count agrees'].sum())} of {len(RESULTS)}
Uploaded              : {', '.join(UPLOADED) or 'nothing'}
Arena commit          : {ARENA_SHA}
Minutes per step      : {', '.join(f'{k} {v / 60:.1f}' for k, v in _STEPS.items())}

Best result per board:
{best[columns].to_string(index=False)}""")
if SMOKE:   # the trial must train a real fighter, predict real sentences and agree with the arena's score
    row = RESULTS.iloc[0]
    assert not FAILED and len(RESULTS) == 1, f"the trial run failed: {FAILED}"
    assert row["own count agrees"], "the own count of correct rows differs from the arena's accuracy"
    assert row["accuracy"] > 0.3, "the fighter got almost no sentence right"

[![NGI0 Commons Fund](../ngi.png)](https://nlnet.nl/project/OpenVoiceOS)

This project was funded through the [NGI0 Commons Fund](https://nlnet.nl/commonsfund), a fund established by [NLnet](https://nlnet.nl) with financial support from the European Commission's [Next Generation Internet](https://ngi.eu) programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No [101135429](https://cordis.europa.eu/project/id/101135429).

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.

Licensed under Apache-2.0.